In [ ]:
# ⚙️ 01 — ENGINE SETUP
# Install dependencies, inspect this Colab runtime, and cache both model weights
# inside the active session. Nothing is written to Google Drive.

import os, sys, time, shutil, pathlib, subprocess, urllib.request, platform

def run_setup(command):
    print("$", " ".join(command))
    subprocess.run(command, check=True)

if shutil.which("ffmpeg") is None or shutil.which("ffprobe") is None:
    run_setup(["apt-get", "-qq", "update"])
    run_setup(["apt-get", "-qq", "install", "-y", "ffmpeg"])

run_setup([
    sys.executable, "-m", "pip", "install", "-q",
    "fastapi", "uvicorn[standard]", "python-multipart",
    "pillow", "opencv-python-headless", "spandrel", "requests"
])

print("\n=== Runtime resources ===")
print("Python:", sys.version.split()[0], "| Platform:", platform.platform())
print("CPU cores:", os.cpu_count() or "unknown")
try:
    import psutil
    print("RAM total:", round(psutil.virtual_memory().total / 1024**3, 1), "GB")
    print("RAM available:", round(psutil.virtual_memory().available / 1024**3, 1), "GB")
except Exception:
    print("RAM details: unavailable")

import torch
print("PyTorch:", torch.__version__)
print("Compute device:", "CUDA GPU — " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("FFmpeg:", subprocess.check_output(["ffmpeg", "-version"], text=True).splitlines()[0])

MODEL_FILES = [
    ("RealESRGAN_x4plus.pth",
     "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",
     40 * 1024 * 1024),
    ("RealESRGAN_x4plus_anime_6B.pth",
     "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth",
     10 * 1024 * 1024),
]

def cache_model(filename, url, minimum_bytes):
    target = pathlib.Path("/content") / filename
    if target.is_file() and target.stat().st_size >= minimum_bytes:
        print("[SESSION CACHE]", filename, f"({target.stat().st_size / 1024**2:.1f} MB)")
        return target
    temporary = target.with_suffix(target.suffix + ".download")
    print("[DOWNLOAD]", filename)
    request = urllib.request.Request(url, headers={"User-Agent": "VeloceAI-Studio/1.0"})
    try:
        with urllib.request.urlopen(request, timeout=240) as response, open(temporary, "wb") as output:
            shutil.copyfileobj(response, output)
        if temporary.stat().st_size < minimum_bytes:
            raise RuntimeError(f"{filename} download is incomplete.")
        temporary.replace(target)
    except Exception:
        temporary.unlink(missing_ok=True)
        raise
    print("[READY]", filename, f"({target.stat().st_size / 1024**2:.1f} MB)")
    return target

for model in MODEL_FILES:
    cache_model(*model)

pathlib.Path("/content/veloceai-session").mkdir(parents=True, exist_ok=True)
print("\n[PASS] Setup complete. Models are cached for this active Colab session.")


In [ ]:
# 🧠 02 — PROCESSING CORE
# Fetch the current API and studio page, start the engine, load both models,
# then run compact real image/video checks before publishing the studio.

import os, sys, time, json, pathlib, subprocess, urllib.request, shutil, io
import requests
from PIL import Image

REPO_RAW = "https://raw.githubusercontent.com/rezeyuri-dot/veloceai/main/"

def fetch_current(filename, destination):
    request = urllib.request.Request(
        REPO_RAW + filename + "?v=" + str(time.time_ns()),
        headers={"Cache-Control": "no-cache", "Pragma": "no-cache"}
    )
    with urllib.request.urlopen(request, timeout=60) as response:
        data = response.read()
    pathlib.Path(destination).write_bytes(data)
    return data

server_code = fetch_current("colab_server.py", "/content/colab_server.py").decode("utf-8")
page_code = fetch_current("index.html", "/content/index.html").decode("utf-8")
required_server = [
    '@app.post("/api/upload/init")',
    '@app.post("/api/upload/chunk")',
    '@app.post("/api/jobs/start")',
    '@app.get("/api/jobs/{job_id}/result")',
    "mobile_optimize:bool=False",
]
missing = [item for item in required_server if item not in server_code]
if missing:
    raise RuntimeError("The downloaded processing core is incomplete: " + ", ".join(missing))
compile(server_code, "/content/colab_server.py", "exec")
if 'data-profile="anime"' not in page_code or 'data-profile="general"' not in page_code:
    raise RuntimeError("The current two-profile studio page was not downloaded.")

# Stop older processes from either this notebook version or the previous launch cell.
for process_name in ("veloceai_tunnel", "veloceai_server", "tunnel", "server"):
    previous = globals().get(process_name)
    try:
        if previous and previous.poll() is None:
            previous.terminate()
            previous.wait(timeout=8)
    except Exception:
        try:
            previous.kill()
        except Exception:
            pass

health = {}
server_log = open("/content/veloceai-server.log", "w")
veloceai_server = subprocess.Popen(
    [sys.executable, "/content/colab_server.py"],
    stdout=server_log, stderr=subprocess.STDOUT
)
LOCAL = "http://127.0.0.1:8000"
for attempt in range(60):
    try:
        health = requests.get(LOCAL + "/health", timeout=3).json()
        if health.get("ok"):
            break
    except Exception:
        pass
    if veloceai_server.poll() is not None:
        break
    time.sleep(2)
if veloceai_server.poll() is not None or not health.get("ok", False):
    print(pathlib.Path("/content/veloceai-server.log").read_text(errors="ignore")[-6000:])
    raise RuntimeError("VeloceAI API did not start successfully.")

# Models are loaded once in this server process and reused for every file in this session.
for profile in ("anime", "general"):
    response = requests.post(LOCAL + "/warmup", params={"profile": profile}, timeout=600)
    response.raise_for_status()
    print("[MODEL READY]", profile.upper(), response.json())

# Tiny synthetic image checks verify actual model inference and exact output dimensions.
test_dir = pathlib.Path("/content/veloceai-self-test")
test_dir.mkdir(exist_ok=True)
sample = Image.new("RGB", (32, 32))
pixels = sample.load()
for y in range(32):
    for x in range(32):
        pixels[x, y] = (x * 8, y * 8, (x ^ y) * 8)
buffer = io.BytesIO()
sample.save(buffer, format="PNG")
image_bytes = buffer.getvalue()

for scale, profile in ((1, "anime"), (2, "general"), (4, "anime")):
    response = requests.post(
        LOCAL + "/api/enhance/image",
        params={"scale": scale, "profile": profile, "denoise": 0,
                "sharpness": 1.0, "line_detail": 0, "anti_alias": 0, "crf": 17},
        files={"file": ("self-test.png", image_bytes, "image/png")},
        timeout=(20, 600)
    )
    response.raise_for_status()
    checked = Image.open(io.BytesIO(response.content))
    expected = (32 * scale, 32 * scale)
    if checked.size != expected:
        raise RuntimeError(f"{profile} {scale}x test returned {checked.size}, expected {expected}.")
    print(f"[PASS] {profile.upper()} {scale}x image:", checked.size, "bytes:", len(response.content))

# Video check confirms output dimensions, original FPS and audio preservation.
video_in = test_dir / "source-with-audio.mp4"
subprocess.run([
    "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
    "-f", "lavfi", "-i", "testsrc2=size=32x32:rate=2:duration=1",
    "-f", "lavfi", "-i", "sine=frequency=660:sample_rate=44100:duration=1",
    "-shortest", "-c:v", "libx264", "-pix_fmt", "yuv420p", "-c:a", "aac", str(video_in)
], check=True)
video_result = requests.post(
    LOCAL + "/api/enhance/video",
    params={"scale": 1, "profile": "anime", "denoise": 0, "sharpness": 1.0,
            "line_detail": 0, "anti_alias": 0, "crf": 25, "mobile_optimize": "true"},
    files={"file": ("self-test.mp4", video_in.read_bytes(), "video/mp4")},
    timeout=(20, 900)
)
video_result.raise_for_status()
video_out = test_dir / "VeloceAI-self-test.mp4"
video_out.write_bytes(video_result.content)
probe = json.loads(subprocess.check_output([
    "ffprobe", "-v", "error", "-show_streams", "-of", "json", str(video_out)
]))
video_streams = [item for item in probe.get("streams", []) if item.get("codec_type") == "video"]
audio_streams = [item for item in probe.get("streams", []) if item.get("codec_type") == "audio"]
if not video_streams or (video_streams[0].get("width"), video_streams[0].get("height")) != (32, 32):
    raise RuntimeError("Video self-test output resolution is incorrect.")
if video_streams[0].get("avg_frame_rate") != "2/1" or not audio_streams:
    raise RuntimeError("Video self-test did not preserve the source FPS and audio.")
if video_streams[0].get("codec_name") != "h264" or video_streams[0].get("profile") != "Main":
    raise RuntimeError("Mobile export did not use the expected H.264 Main profile.")
if audio_streams[0].get("codec_name") != "aac":
    raise RuntimeError("MP4 export audio codec is not AAC.")
print("[PASS] MP4 export: H.264 Main, 32x32, 2 FPS and AAC audio.")
print("\n[READY] Processing core is running at", LOCAL)
print("Keep this Colab runtime active while using VeloceAI.")


In [ ]:
# 🚀 03 — LAUNCH VELOCEAI
# Publish the local API with a temporary Cloudflare Quick Tunnel and embed
# the studio directly in this Colab output.

import re, time, pathlib, platform, stat, shutil, subprocess, requests, urllib.request
from IPython.display import display, IFrame

if "veloceai_server" not in globals() or veloceai_server.poll() is not None:
    raise RuntimeError("Run cell 02 — PROCESSING CORE first.")
requests.get("http://127.0.0.1:8000/health", timeout=5).raise_for_status()

cloudflared = shutil.which("cloudflared") or "/usr/local/bin/cloudflared"
if not pathlib.Path(cloudflared).exists():
    architecture = platform.machine().lower()
    binary = {
        "x86_64": "cloudflared-linux-amd64",
        "amd64": "cloudflared-linux-amd64",
        "aarch64": "cloudflared-linux-arm64",
        "arm64": "cloudflared-linux-arm64",
    }.get(architecture)
    if not binary:
        raise RuntimeError("Unsupported Colab CPU architecture for Cloudflare Tunnel: " + architecture)
    download_url = "https://github.com/cloudflare/cloudflared/releases/latest/download/" + binary
    print("Downloading Cloudflare Tunnel...")
    temporary = "/content/cloudflared.download"
    urllib.request.urlretrieve(download_url, temporary)
    pathlib.Path(temporary).replace(cloudflared)
    pathlib.Path(cloudflared).chmod(
        pathlib.Path(cloudflared).stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH
    )

previous = globals().get("veloceai_tunnel")
try:
    if previous and previous.poll() is None:
        previous.terminate()
        previous.wait(timeout=8)
except Exception:
    pass

tunnel_log = open("/content/veloceai-tunnel.log", "w")
veloceai_tunnel = subprocess.Popen(
    [cloudflared, "tunnel", "--protocol", "http2", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=tunnel_log, stderr=subprocess.STDOUT
)
studio_url = None
for attempt in range(60):
    time.sleep(2)
    log = pathlib.Path("/content/veloceai-tunnel.log").read_text(errors="ignore")
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log)
    if match:
        studio_url = match.group(0)
        break
    if veloceai_tunnel.poll() is not None:
        break
if not studio_url:
    print(pathlib.Path("/content/veloceai-tunnel.log").read_text(errors="ignore")[-5000:])
    raise RuntimeError("Cloudflare did not provide a public studio URL.")

last_error = None
for attempt in range(24):
    try:
        health_response = requests.get(studio_url + "/health", timeout=15)
        health_response.raise_for_status()
        page_response = requests.get(studio_url + "/", timeout=15)
        page_response.raise_for_status()
        if "VeloceAI Studio" not in page_response.text:
            raise RuntimeError("The public URL is not serving the current VeloceAI Studio page.")
        break
    except Exception as error:
        last_error = error
        time.sleep(5)
else:
    raise RuntimeError("The tunnel was created but failed its public check: " + repr(last_error))

print("VeloceAI Studio is live:", studio_url)
print("Engine status:", health_response.json())
display(IFrame(src=studio_url, width="100%", height=900))
print("Leave this Colab runtime active while processing media.")
